# Predicting customer churn with Python

In [ ]:
# Loading the data
import pandas as pd
import numpy as np
import janitor
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

df = pd.read_excel('../../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.xlsx', sheet_name=0).clean_names(case_type='snake')


In [ ]:
df.shape

In [ ]:
df.head(10)

# Data Analysis & Preparation

#### - Encoding target var: Churn

In [ ]:
df['churn'] = df['churn'].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
df['churn'].mean()

#### - TotalCharges

In [ ]:
df['total_charges'] = df['total_charges'].replace(' ', np.nan).astype(float)

In [ ]:
df.shape

In [ ]:
df.dropna().shape

In [ ]:
df = df.dropna()

#### - Continuous Vars

In [ ]:
df[['tenure', 'monthly_charges', 'total_charges']].describe()

In [ ]:
df['monthly_charges'] = np.log(df['monthly_charges'])
df['monthly_charges'] = (df['monthly_charges'] - df['monthly_charges'].mean())/df['monthly_charges'].std()

df['total_charges'] = np.log(df['total_charges'])
df['total_charges'] = (df['total_charges'] - df['total_charges'].mean())/df['total_charges'].std()

df['tenure'] = (df['tenure'] - df['tenure'].mean())/df['tenure'].std()

In [ ]:
df[['tenure', 'monthly_charges', 'total_charges']].describe()

In [ ]:
continuous_vars = list(df.describe().columns)
continuous_vars

#### - One-Hot Encoding

In [ ]:
for col in list(df.columns):
    print(col, df[col].nunique())

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.templates.default = "ggplot2"

gender_counts = df.groupby("gender", as_index=False)["customer_id"].count()
fig = px.bar(gender_counts, x="gender", y="customer_id", title="Gender", color_discrete_sequence=["skyblue"])
fig.update_layout(xaxis_title="gender", yaxis_title="count")
fig.show()

internet_counts = df.groupby("internet_service", as_index=False)["customer_id"].count()
fig = px.bar(internet_counts, x="internet_service", y="customer_id", title="Internet Service", color_discrete_sequence=["skyblue"])
fig.update_layout(xaxis_title="internet_service", yaxis_title="count")
fig.show()

payment_counts = df.groupby("payment_method", as_index=False)["customer_id"].count()
fig = px.bar(payment_counts, x="payment_method", y="customer_id", title="Payment Method", color_discrete_sequence=["skyblue"])
fig.update_layout(xaxis_title="payment_method", yaxis_title="count")
fig.show()

In [ ]:
dummy_cols = []

sample_set = df[['tenure', 'monthly_charges', 'total_charges', 'churn']].copy(deep=True)

for col in list(df.columns):
    if col not in ['tenure', 'monthly_charges', 'total_charges', 'churn'] and df[col].nunique() < 5:
        dummy_vars = pd.get_dummies(df[col])
        # dummy_vars.columns = [col+str(x) for x in dummy_vars.columns]
        dummy_vars.columns = [col + '_' + str(x).replace(" ", "_").replace("-", "_").lower() for x in dummy_vars.columns]
        sample_set = pd.concat([sample_set, dummy_vars], axis=1)

In [ ]:
sample_set.head(10)

In [ ]:
sample_set.shape

In [ ]:
list(sample_set.columns)

# 3. Train & Test Sets

In [ ]:
target_var = 'churn'
features = [x for x in list(sample_set.columns) if x != target_var]

# 4. Aritificial Neural Network (ANN) with Keras

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense

#### - Training a Neural Network Model

In [ ]:
model = Sequential([
    Input(shape=(len(features),)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    sample_set[features], 
    sample_set[target_var], 
    test_size=0.3,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train, epochs=50, batch_size=100)

#### - Accuracy, Precision, Recall

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [ ]:
in_sample_preds = [round(x[0]) for x in model.predict(X_train)]
out_sample_preds = [round(x[0]) for x in model.predict(X_test)]

In [ ]:
print('In-Sample Accuracy: %0.4f' % accuracy_score(y_train, in_sample_preds))
print('Out-of-Sample Accuracy: %0.4f' % accuracy_score(y_test, out_sample_preds))

print('\n')

print('In-Sample Precision: %0.4f' % precision_score(y_train, in_sample_preds))
print('Out-of-Sample Precision: %0.4f' % precision_score(y_test, out_sample_preds))

print('\n')

print('In-Sample Recall: %0.4f' % recall_score(y_train, in_sample_preds))
print('Out-of-Sample Recall: %0.4f' % recall_score(y_test, out_sample_preds))

#### - ROC & AUC

In [ ]:
from sklearn.metrics import roc_curve, auc

In [ ]:
in_sample_preds = [x[0] for x in model.predict(X_train)]
out_sample_preds = [x[0] for x in model.predict(X_test)]

In [ ]:
in_sample_fpr, in_sample_tpr, in_sample_thresholds = roc_curve(y_train, in_sample_preds)
out_sample_fpr, out_sample_tpr, out_sample_thresholds = roc_curve(y_test, out_sample_preds)

In [ ]:
in_sample_roc_auc = auc(in_sample_fpr, in_sample_tpr)
out_sample_roc_auc = auc(out_sample_fpr, out_sample_tpr)

print('In-Sample AUC: %0.4f' % in_sample_roc_auc)
print('Out-Sample AUC: %0.4f' % out_sample_roc_auc)

In [ ]:
fig = px.line(title='ROC Curve')

fig.add_scatter(x=out_sample_fpr, y=out_sample_tpr, mode='lines', name='Out-Sample ROC curve (area = %0.4f)' % out_sample_roc_auc, line=dict(color='darkorange'))
fig.add_scatter(x=in_sample_fpr, y=in_sample_tpr, mode='lines', name='In-Sample ROC curve (area = %0.4f)' % in_sample_roc_auc, line=dict(color='navy'))
fig.add_scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random', line=dict(color='gray', width=1, dash='dash'))

fig.update_layout(
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0.0, 1.0]),
    yaxis=dict(range=[0.0, 1.05]),
    legend=dict(x=0.6, y=0.1),
    width=800,
    height=600
)

fig.show()
